# ⭐ Day 76: Multi-Agent Reinforcement Learning & Advanced Topics
## Cooperation, Competition, and the Future of RL
### Python & AI Learning Path — Day 76 of 369


Welcome to **Day 76** of your Python & AI Learning Path! 🎉 Today marks a significant milestone as we conclude our deep dive into Reinforcement Learning (RL). Over the past sessions, we have journeyed from the fundamentals of Q-Learning and policy gradients to deep neural network-based methods like DQN and PPO. Now, we step into one of the most exciting frontiers of AI: **Multi-Agent Reinforcement Learning (MARL)**.

In the real world, intelligence rarely operates in isolation. Self-driving cars must negotiate intersections with each other, robotic teams coordinate to build structures, game AI adapts to human and artificial opponents, and financial trading agents react to market dynamics shaped by thousands of other agents. MARL studies how multiple autonomous entities learn to make decisions in shared environments, where they may need to **cooperate**, **compete**, or do both simultaneously.

Beyond MARL, we will explore critical advanced topics that bridge classical RL and cutting-edge research. These include **Hierarchical RL** (learning at multiple levels of abstraction), **Model-Based RL** (planning with learned world models), **Offline RL** (learning from fixed datasets without environment interaction), and **Meta-RL** (learning to learn). Understanding these concepts will prepare you for the state-of-the-art research and applications driving the field forward.

This notebook is designed to be both conceptual and practical. We will build a simple multi-agent environment, visualize agent interactions, and discuss the algorithms that power modern MARL systems. Whether you aspire to build autonomous swarms, competitive game AI, or robust trading systems, the concepts covered today will provide a strong foundation. Let's dive in! 🚀


## 📚 Table of Contents

1. [From Single-Agent to Multi-Agent Reinforcement Learning](#1-from-single-agent-to-multi-agent-reinforcement-learning)
2. [Key Challenges in Multi-Agent Settings](#2-key-challenges-in-multi-agent-settings)
3. [Types of Multi-Agent Scenarios](#3-types-of-multi-agent-scenarios)
4. [Simple Multi-Agent Example: Two-Agent Grid World](#4-simple-multi-agent-example-two-agent-grid-world)
5. [Independent Q-Learning vs Joint Action Learning](#5-independent-q-learning-vs-joint-action-learning)
6. [Modern Approaches (QMIX, MADDPG, MAT)](#6-modern-approaches)
7. [Advanced Single-Agent Topics](#7-advanced-single-agent-topics)
8. [Real-World Applications of Multi-Agent RL](#8-real-world-applications-of-multi-agent-rl)
9. [Current Limitations and Future Directions](#9-current-limitations-and-future-directions)
10. [🛠️ Hands-On Exercises](#10-hands-on-exercises)
11. [Solutions & Discussion](#11-solutions--discussion)


## 1. From Single-Agent to Multi-Agent Reinforcement Learning

Single-agent RL assumes one agent interacting with a stationary environment. The Markov Decision Process (MDP) is defined as $(S, A, P, R, \gamma)$. In MARL, we extend this to a **Stochastic Game** (or Markov Game) defined by $(N, S, \{A_i\}, P, \{R_i\}, \gamma)$, where $N$ is the number of agents.

Key differences:
- **Non-stationarity**: The environment becomes non-stationary from any single agent's perspective because other agents are learning and changing their policies.
- **Joint action space**: The transition and reward functions depend on the joint actions of all agents.
- **Partial observability**: Agents often observe only local information, leading to Dec-POMDP formulations.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
import random
import seaborn as sns
from IPython.display import clear_output
import time

# Set style for beautiful visualizations
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
np.random.seed(76)
random.seed(76)

print("✅ Libraries imported successfully!")
print("🎯 Ready to explore Multi-Agent Reinforcement Learning!")


## 2. Key Challenges in Multi-Agent Settings

### 🔄 Non-Stationarity
As other agents learn, the environment dynamics change. What was optimal yesterday may not be optimal today. This breaks the fundamental assumption of most single-agent RL algorithms.

### 🎯 Credit Assignment
In cooperative settings with shared rewards, determining which agent contributed to success is notoriously difficult (the "multi-agent credit assignment problem").

### 📡 Communication
Agents must decide what to communicate, when, and to whom. Learning communication protocols from scratch is an active research area.

### 🧮 Scalability
The joint action space grows exponentially with the number of agents ($|A|^N$), making centralized approaches intractable.


In [ ]:
# Visualizing the curse of dimensionality in joint action spaces
agents = np.arange(1, 11)
actions_per_agent = 5
joint_actions = actions_per_agent ** agents

fig, ax = plt.subplots(figsize=(10, 6))
ax.semilogy(agents, joint_actions, 'o-', linewidth=3, markersize=8, color='#e74c3c')
ax.fill_between(agents, joint_actions, alpha=0.3, color='#e74c3c')
ax.set_xlabel('Number of Agents', fontsize=14, fontweight='bold')
ax.set_ylabel('Joint Action Space Size (log scale)', fontsize=14, fontweight='bold')
ax.set_title('🔥 The Curse of Dimensionality in MARL\nJoint Action Space Explosion', fontsize=16, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.set_xticks(agents)

for i, txt in enumerate(joint_actions[:6]):
    ax.annotate(f'{int(txt)}', (agents[i], joint_actions[i]), 
                textcoords="offset points", xytext=(0, 10), ha='center', fontsize=10)

plt.tight_layout()
plt.show()
print("💡 Notice how quickly the joint action space becomes intractable!")


## 3. Types of Multi-Agent Scenarios

| Scenario | Reward Structure | Example |
|----------|-----------------|---------|
| **Cooperative** | Shared reward $R_1 = R_2 = ... = R_N$ | Robot soccer team scoring a goal |
| **Competitive** | Zero-sum $\sum R_i = 0$ | Chess, Poker, Adversarial games |
| **Mixed** | General-sum (arbitrary) | Autonomous driving, Market trading |

Let's visualize these interaction structures.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
scenarios = [
    ('🤝 Cooperative', 'Shared\nReward', '#2ecc71', 'Agents work together\ntoward common goal'),
    ('⚔️ Competitive', 'Zero-Sum\nReward', '#e74c3c', "One agent's gain is\nanother's loss"),
    ('🌐 Mixed', 'General-Sum\nReward', '#f39c12', 'Complex interdependencies\nand externalities')
]

for ax, (title, reward, color, desc) in zip(axes, scenarios):
    # Draw agents as circles
    circle1 = plt.Circle((0.3, 0.5), 0.12, color=color, alpha=0.7)
    circle2 = plt.Circle((0.7, 0.5), 0.12, color=color, alpha=0.7)
    ax.add_patch(circle1)
    ax.add_patch(circle2)
    
    # Draw connection
    if 'Cooperative' in title:
        ax.annotate('', xy=(0.58, 0.5), xytext=(0.42, 0.5),
                   arrowprops=dict(arrowstyle='<->', color=color, lw=3))
    elif 'Competitive' in title:
        ax.annotate('', xy=(0.58, 0.5), xytext=(0.42, 0.5),
                   arrowprops=dict(arrowstyle='<->', color=color, lw=3, linestyle='--'))
    else:
        ax.annotate('', xy=(0.58, 0.5), xytext=(0.42, 0.5),
                   arrowprops=dict(arrowstyle='<->', color=color, lw=2))
        ax.annotate('', xy=(0.42, 0.5), xytext=(0.58, 0.5),
                   arrowprops=dict(arrowstyle='<->', color='#3498db', lw=2))
    
    ax.text(0.5, 0.85, title, ha='center', fontsize=14, fontweight='bold')
    ax.text(0.5, 0.2, desc, ha='center', fontsize=10, style='italic')
    ax.text(0.5, 0.05, reward, ha='center', fontsize=11, fontweight='bold', color=color)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_aspect('equal')
    ax.axis('off')

plt.suptitle('Multi-Agent Interaction Structures', fontsize=18, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


## 4. Simple Multi-Agent Example: Two-Agent Grid World

Let's build a simple cooperative grid world where two agents must simultaneously reach a shared goal. They receive a shared reward only when BOTH are on the goal at the same time. This requires coordination!


In [ ]:
class TwoAgentGridWorld:
    """
    A 5x5 grid world for two agents.
    Agents must BOTH reach the goal to receive reward.
    """
    def __init__(self, size=5, max_steps=50):
        self.size = size
        self.max_steps = max_steps
        self.reset()
    
    def reset(self):
        self.agent1_pos = np.array([0, 0])
        self.agent2_pos = np.array([self.size-1, 0])
        self.goal = np.array([self.size//2, self.size-1])
        self.steps = 0
        return self._get_obs()
    
    def _get_obs(self):
        return (tuple(self.agent1_pos), tuple(self.agent2_pos))
    
    def step(self, action1, action2):
        # Actions: 0=Up, 1=Down, 2=Left, 3=Right
        moves = {0: (-1, 0), 1: (1, 0), 2: (0, -1), 3: (0, 1)}
        
        for agent_pos, action in [(self.agent1_pos, action1), (self.agent2_pos, action2)]:
            move = moves[action]
            new_pos = agent_pos + np.array(move)
            # Boundary check
            new_pos = np.clip(new_pos, 0, self.size - 1)
            agent_pos[:] = new_pos
        
        self.steps += 1
        
        # Cooperative reward: both must be on goal
        goal_reached = np.array_equal(self.agent1_pos, self.goal) and np.array_equal(self.agent2_pos, self.goal)
        reward = 10.0 if goal_reached else -0.1
        done = goal_reached or self.steps >= self.max_steps
        
        return self._get_obs(), reward, done
    
    def render(self):
        grid = np.zeros((self.size, self.size))
        grid[self.goal[0], self.goal[1]] = 0.5  # Goal
        grid[self.agent1_pos[0], self.agent1_pos[1]] = 0.8  # Agent 1
        grid[self.agent2_pos[0], self.agent2_pos[1]] = 1.0  # Agent 2
        return grid

# Test environment
env = TwoAgentGridWorld()
obs = env.reset()
print(f"🌍 Grid World initialized!")
print(f"📍 Agent 1 starts at: {obs[0]}")
print(f"📍 Agent 2 starts at: {obs[1]}")
print(f"🎯 Goal is at: {env.goal}")


In [ ]:
# Visualize the initial state
fig, ax = plt.subplots(figsize=(6, 6))
grid = env.render()
im = ax.imshow(grid, cmap='RdYlGn', vmin=0, vmax=1)
ax.set_title('🗺️ Two-Agent Cooperative Grid World', fontsize=14, fontweight='bold')
ax.set_xticks(np.arange(-0.5, env.size, 1), minor=True)
ax.set_yticks(np.arange(-0.5, env.size, 1), minor=True)
ax.grid(which='minor', color='black', linestyle='-', linewidth=2)
ax.tick_params(which='minor', size=0)
ax.set_xticks([])
ax.set_yticks([])

# Add labels
ax.text(env.goal[1], env.goal[0], '🎯', ha='center', va='center', fontsize=20)
ax.text(env.agent1_pos[1], env.agent1_pos[0], '🤖', ha='center', va='center', fontsize=20)
ax.text(env.agent2_pos[1], env.agent2_pos[0], '👾', ha='center', va='center', fontsize=20)
plt.show()


## 5. Independent Q-Learning vs Joint Action Learning

### Independent Q-Learning (IQL)
Each agent learns its own Q-function treating other agents as part of the environment. Simple but suffers from non-stationarity.

### Joint Action Learning (JAL)
Learns Q(s, a₁, a₂, ..., aₙ) — the joint action space. Avoids non-stationarity but suffers from exponential complexity.


In [ ]:
class IndependentQLearningAgent:
    """Agent using Independent Q-Learning (IQL)"""
    def __init__(self, agent_id, state_size, n_actions, alpha=0.1, gamma=0.95, epsilon=1.0):
        self.agent_id = agent_id
        self.n_actions = n_actions
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon
        self.q_table = defaultdict(lambda: np.zeros(n_actions))
        self.episode_rewards = []
    
    def select_action(self, state):
        if random.random() < self.epsilon:
            return random.randint(0, self.n_actions - 1)
        return np.argmax(self.q_table[state])
    
    def update(self, state, action, reward, next_state, done):
        current_q = self.q_table[state][action]
        max_next_q = np.max(self.q_table[next_state]) if not done else 0
        self.q_table[state][action] = current_q + self.alpha * (reward + self.gamma * max_next_q - current_q)
    
    def decay_epsilon(self, decay=0.995, min_eps=0.01):
        self.epsilon = max(min_eps, self.epsilon * decay)

print("🧠 Independent Q-Learning Agent class defined!")
print("📊 Each agent maintains its own Q-table and treats others as environment dynamics.")


In [ ]:
# Training two independent Q-learning agents in the cooperative grid world
env = TwoAgentGridWorld(size=5, max_steps=50)
agent1 = IndependentQLearningAgent(1, env.size**2, 4, alpha=0.2, gamma=0.95)
agent2 = IndependentQLearningAgent(2, env.size**2, 4, alpha=0.2, gamma=0.95)

n_episodes = 2000
episode_rewards = []
success_rate_window = []
coordination_success = []

print("🏃 Training Independent Q-Learning Agents...")
for episode in range(n_episodes):
    state = env.reset()
    total_reward = 0
    done = False
    
    while not done:
        action1 = agent1.select_action(state)
        action2 = agent2.select_action(state)
        
        next_state, reward, done = env.step(action1, action2)
        total_reward += reward
        
        # Independent updates (each sees the shared reward)
        agent1.update(state, action1, reward, next_state, done)
        agent2.update(state, action2, reward, next_state, done)
        
        state = next_state
    
    episode_rewards.append(total_reward)
    success_rate_window.append(1 if total_reward > 0 else 0)
    coordination_success.append(np.mean(success_rate_window[-100:]) if len(success_rate_window) >= 100 else np.mean(success_rate_window))
    
    agent1.decay_epsilon(0.998, 0.05)
    agent2.decay_epsilon(0.998, 0.05)
    
    if (episode + 1) % 500 == 0:
        print(f"  Episode {episode+1}/{n_episodes} | Avg Reward (last 100): {np.mean(episode_rewards[-100:]):.2f} | Success Rate: {coordination_success[-1]:.2%}")

print("✅ Training complete!")


In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Episode rewards
window = 50
smoothed_rewards = np.convolve(episode_rewards, np.ones(window)/window, mode='valid')
axes[0].plot(episode_rewards, alpha=0.3, color='gray', label='Raw')
axes[0].plot(range(window-1, len(episode_rewards)), smoothed_rewards, color='#3498db', linewidth=2, label=f'Moving Avg ({window})')
axes[0].axhline(y=10, color='green', linestyle='--', alpha=0.7, label='Max Reward')
axes[0].set_xlabel('Episode', fontsize=12)
axes[0].set_ylabel('Total Reward', fontsize=12)
axes[0].set_title('📈 Training Rewards: Independent Q-Learning', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Success rate
axes[1].plot(coordination_success, color='#e74c3c', linewidth=2)
axes[1].axhline(y=1.0, color='green', linestyle='--', alpha=0.7, label='Perfect Coordination')
axes[1].set_xlabel('Episode', fontsize=12)
axes[1].set_ylabel('Success Rate (100-episode window)', fontsize=12)
axes[1].set_title('🎯 Coordination Success Rate', fontsize=14, fontweight='bold')
axes[1].set_ylim(0, 1.05)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print(f"🎉 Final coordination success rate: {coordination_success[-1]:.2%}")


In [ ]:
# Visualize learned policy trajectories
def visualize_policy(env, agent1, agent2, n_trajectories=3):
    fig, axes = plt.subplots(1, n_trajectories, figsize=(15, 5))
    
    for traj_idx, ax in enumerate(axes):
        state = env.reset()
        trajectory1 = [env.agent1_pos.copy()]
        trajectory2 = [env.agent2_pos.copy()]
        done = False
        steps = 0
        
        while not done and steps < env.max_steps:
            # Greedy action selection
            old_eps1, old_eps2 = agent1.epsilon, agent2.epsilon
            agent1.epsilon, agent2.epsilon = 0, 0
            action1 = agent1.select_action(state)
            action2 = agent2.select_action(state)
            agent1.epsilon, agent2.epsilon = old_eps1, old_eps2
            
            state, _, done = env.step(action1, action2)
            trajectory1.append(env.agent1_pos.copy())
            trajectory2.append(env.agent2_pos.copy())
            steps += 1
        
        # Plot
        grid = np.zeros((env.size, env.size))
        ax.imshow(grid, cmap='Blues', alpha=0.3)
        ax.set_xticks(np.arange(-0.5, env.size, 1), minor=True)
        ax.set_yticks(np.arange(-0.5, env.size, 1), minor=True)
        ax.grid(which='minor', color='black', linestyle='-', linewidth=1)
        ax.tick_params(which='minor', size=0)
        ax.set_xticks([])
        ax.set_yticks([])
        
        # Plot trajectories
        traj1 = np.array(trajectory1)
        traj2 = np.array(trajectory2)
        ax.plot(traj1[:, 1], traj1[:, 0], 'o-', color='#3498db', linewidth=2, markersize=6, label='Agent 1')
        ax.plot(traj2[:, 1], traj2[:, 0], 's-', color='#e74c3c', linewidth=2, markersize=6, label='Agent 2')
        
        # Markers
        ax.text(env.goal[1], env.goal[0], '🎯', ha='center', va='center', fontsize=20)
        ax.text(trajectory1[0][1], trajectory1[0][0], '🤖', ha='center', va='center', fontsize=16)
        ax.text(trajectory2[0][1], trajectory2[0][0], '👾', ha='center', va='center', fontsize=16)
        
        success = np.array_equal(env.agent1_pos, env.goal) and np.array_equal(env.agent2_pos, env.goal)
        title_color = 'green' if success else 'red'
        ax.set_title(f'Trajectory {traj_idx+1}\n{"✅ Success" if success else "❌ Failed"}', 
                    fontsize=12, fontweight='bold', color=title_color)
        ax.legend(loc='upper left')
        ax.invert_yaxis()
    
    plt.suptitle('🗺️ Learned Agent Trajectories (Greedy Policy)', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()

visualize_policy(env, agent1, agent2)


## 6. Modern Approaches

### QMIX (2018)
Learns a joint action-value function $Q_{tot}$ that factors into individual $Q_i$ values via a monotonic mixing network. Allows centralized training with decentralized execution (CTDE).

### MADDPG (2017)
Multi-Agent extension of DDPG. Uses centralized critics (access to all agents' observations/actions) but decentralized actors. Handles continuous action spaces.

### MAT / MAPPO (2021+)
Multi-Agent Transformer and Multi-Agent PPO. Leverages attention mechanisms and modern policy gradient methods for scalable MARL.

### Value Decomposition Networks (VDN)
Simpler than QMIX: $Q_{tot} = \sum_i Q_i$. Works well for fully cooperative tasks.


In [ ]:
# Conceptual diagram: Centralized Training with Decentralized Execution (CTDE)
fig, ax = plt.subplots(figsize=(12, 8))
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis('off')

# Centralized training box
training_box = plt.Rectangle((1, 5.5), 8, 3.5, fill=True, facecolor='#d5f5e3', edgecolor='#27ae60', linewidth=3, linestyle='--')
ax.add_patch(training_box)
ax.text(5, 8.7, 'CENTRALIZED TRAINING', ha='center', fontsize=14, fontweight='bold', color='#27ae60')
ax.text(5, 8.2, '(Global State + All Observations + All Actions)', ha='center', fontsize=10, style='italic', color='#27ae60')

# Central mixer
mixer = plt.Circle((5, 6.8), 0.6, color='#27ae60', alpha=0.8)
ax.add_patch(mixer)
ax.text(5, 6.8, 'Mixer\n(QMIX/VDN)', ha='center', va='center', fontsize=9, fontweight='bold', color='white')

# Agent networks
agent_positions = [(2.5, 6.8), (7.5, 6.8)]
agent_labels = ['Agent 1\nQ-Network', 'Agent 2\nQ-Network']
for pos, label in zip(agent_positions, agent_labels):
    box = plt.Rectangle((pos[0]-0.6, pos[1]-0.4), 1.2, 0.8, fill=True, facecolor='#3498db', edgecolor='#2980b9', linewidth=2)
    ax.add_patch(box)
    ax.text(pos[0], pos[1], label, ha='center', va='center', fontsize=8, fontweight='bold', color='white')
    # Arrows to mixer
    ax.annotate('', xy=(5, 6.8), xytext=(pos[0] + (0.6 if pos[0] < 5 else -0.6), 6.8),
               arrowprops=dict(arrowstyle='->', color='#27ae60', lw=2))

# Decentralized execution box
exec_box = plt.Rectangle((1, 1), 8, 3.5, fill=True, facecolor='#fdebd0', edgecolor='#e67e22', linewidth=3, linestyle='--')
ax.add_patch(exec_box)
ax.text(5, 4.2, 'DECENTRALIZED EXECUTION', ha='center', fontsize=14, fontweight='bold', color='#e67e22')
ax.text(5, 3.7, '(Each agent acts using only its local observation)', ha='center', fontsize=10, style='italic', color='#e67e22')

# Execution agents
exec_positions = [(2.5, 2.5), (7.5, 2.5)]
exec_labels = ['Agent 1\nPolicy', 'Agent 2\nPolicy']
for pos, label in zip(exec_positions, exec_labels):
    box = plt.Rectangle((pos[0]-0.6, pos[1]-0.4), 1.2, 0.8, fill=True, facecolor='#e74c3c', edgecolor='#c0392b', linewidth=2)
    ax.add_patch(box)
    ax.text(pos[0], pos[1], label, ha='center', va='center', fontsize=8, fontweight='bold', color='white')
    # Local observation arrows
    ax.annotate('', xy=(pos[0], 2.9), xytext=(pos[0], 1.3),
               arrowprops=dict(arrowstyle='->', color='#e67e22', lw=2))
    ax.text(pos[0], 1.1, 'Local Obs', ha='center', fontsize=8, color='#e67e22')

# Arrow from training to execution
ax.annotate('', xy=(5, 3.7), xytext=(5, 5.5),
           arrowprops=dict(arrowstyle='->', color='black', lw=3, linestyle=':'))
ax.text(5.3, 4.6, 'Trained\nPolicies', ha='left', fontsize=9, fontweight='bold')

plt.title('🏗️ Centralized Training with Decentralized Execution (CTDE)', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()


## 7. Advanced Single-Agent Topics

These topics represent the cutting edge of RL research and are increasingly relevant for building robust, sample-efficient agents.


In [ ]:
# Advanced Topics Overview Visualization
topics = [
    ('🏗️ Hierarchical RL', 'Learn policies at multiple\ntime scales (options/skills)', '#9b59b6'),
    ('🧠 Model-Based RL', 'Learn environment model\nfor planning (Dreamer, MuZero)', '#3498db'),
    ('📦 Offline RL', 'Learn from fixed datasets\nwithout exploration (CQL, IQL)', '#e74c3c'),
    ('🔄 Meta-RL', 'Learn to learn quickly\nfrom few experiences (MAML, RL²)', '#2ecc71'),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, (title, desc, color) in enumerate(topics):
    ax = axes[idx]
    
    # Background
    rect = plt.Rectangle((0.05, 0.05), 0.9, 0.9, fill=True, facecolor=color, alpha=0.15, 
                         edgecolor=color, linewidth=3, transform=ax.transAxes)
    ax.add_patch(rect)
    
    ax.text(0.5, 0.75, title, transform=ax.transAxes, ha='center', fontsize=16, 
            fontweight='bold', color=color)
    ax.text(0.5, 0.45, desc, transform=ax.transAxes, ha='center', fontsize=11, 
            style='italic', color='black')
    
    # Key algorithms
    if 'Hierarchical' in title:
        algos = 'Options Framework\nFeUdal Networks\nHIRO'
    elif 'Model-Based' in title:
        algos = 'MuZero\nDreamerV3\nMBPO\nPlaNet'
    elif 'Offline' in title:
        algos = 'CQL\nIQL\nDecision Transformer\nAWAC'
    else:
        algos = 'MAML\nRL²\nProMP\nPEARL'
    
    ax.text(0.5, 0.2, f'Key Methods:\n{algos}', transform=ax.transAxes, ha='center', 
            fontsize=9, fontweight='bold', color='darkslategray')
    
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')

plt.suptitle('🚀 Advanced RL Topics Bridging Research and Practice', fontsize=18, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


In [ ]:
# Hierarchical RL Concept: Options Framework
fig, ax = plt.subplots(figsize=(12, 6))

# Timeline
ax.plot([0, 10], [0, 0], 'k-', linewidth=2)

# Primitive actions
primitive_times = [1, 2, 3, 4, 5, 6, 7, 8, 9]
for t in primitive_times:
    ax.plot([t, t], [-0.1, 0.1], 'k-', linewidth=2)
    ax.text(t, -0.3, f'a{t}', ha='center', fontsize=10)

ax.text(5, -0.7, 'Primitive Actions (Standard RL)', ha='center', fontsize=12, fontweight='bold')

# Options (temporal abstraction)
option_colors = ['#e74c3c', '#3498db', '#2ecc71']
options = [(0.5, 3.5, 'Option 1:\nGo to Door'), (3.5, 6.5, 'Option 2:\nOpen Door'), (6.5, 9.5, 'Option 3:\nGo Through')]
for i, (start, end, label) in enumerate(options):
    rect = plt.Rectangle((start, 0.5), end-start, 0.8, fill=True, 
                         facecolor=option_colors[i], alpha=0.4, edgecolor=option_colors[i], linewidth=2)
    ax.add_patch(rect)
    ax.text((start+end)/2, 0.9, label, ha='center', va='center', fontsize=10, fontweight='bold', color=option_colors[i])

ax.text(5, 1.8, 'Options (Hierarchical RL)', ha='center', fontsize=12, fontweight='bold')

ax.set_xlim(0, 10)
ax.set_ylim(-1, 2.5)
ax.axis('off')
ax.set_title('⏱️ Temporal Abstraction in Hierarchical RL', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()
print("💡 Options allow agents to reason at higher levels of abstraction, dramatically improving learning efficiency!")


## 8. Real-World Applications of Multi-Agent RL

Multi-Agent RL is transitioning from research to production across diverse domains:


In [ ]:
# Real-world applications visualization
applications = [
    ('🚗 Autonomous Driving', 'Multi-vehicle coordination\nIntersection negotiation\nPlatooning', '#3498db', 95),
    ('🎮 Game AI', 'Dota 2 (OpenAI Five)\nStarCraft II (AlphaStar)\nStrategic reasoning', '#9b59b6', 90),
    ('🤖 Robotics', 'Swarm robotics\nWarehouse automation\nSearch and rescue teams', '#e74c3c', 85),
    ('💹 Financial Markets', 'Algorithmic trading\nMarket making\nPortfolio management', '#2ecc71', 80),
    ('⚡ Smart Grids', 'Energy distribution\nDemand response\nMicrogrid management', '#f39c12', 75),
    ('📡 Network Routing', 'Traffic optimization\nPacket routing\nLoad balancing', '#1abc9c', 70),
]

fig, ax = plt.subplots(figsize=(12, 8))
y_positions = np.arange(len(applications))

for idx, (app, desc, color, maturity) in enumerate(applications):
    # Bar
    bar = ax.barh(idx, maturity, color=color, alpha=0.7, height=0.6, edgecolor='black', linewidth=1)
    
    # Application name
    ax.text(2, idx, app, va='center', fontsize=12, fontweight='bold', color='white')
    
    # Description
    ax.text(maturity + 2, idx, desc, va='center', fontsize=9, style='italic')
    
    # Maturity score
    ax.text(maturity - 5, idx, f'{maturity}%', va='center', ha='right', fontsize=10, fontweight='bold', color='white')

ax.set_yticks(y_positions)
ax.set_yticklabels([])
ax.set_xlim(0, 110)
ax.set_xlabel('Technology Maturity / Adoption Index', fontsize=12, fontweight='bold')
ax.set_title('🌍 Real-World MARL Applications & Maturity', fontsize=16, fontweight='bold')
ax.grid(True, axis='x', alpha=0.3)
ax.invert_yaxis()

# Legend
ax.text(50, -0.8, 'Maturity Scale: Research → Pilot → Production', ha='center', fontsize=10, style='italic', color='gray')

plt.tight_layout()
plt.show()


## 9. Current Limitations and Future Directions

### ⚠️ Key Limitations
1. **Sample Efficiency**: MARL requires orders of magnitude more samples than single-agent RL.
2. **Scalability**: Current methods struggle beyond ~10-20 agents.
3. **Generalization**: Agents often overfit to specific teammates or opponents.
4. **Safety**: Multi-agent systems can exhibit emergent, unpredictable behaviors.
5. **Evaluation**: Lack of standardized benchmarks and evaluation protocols.

### 🔮 Future Directions
- **Foundation Models for RL**: Leveraging LLMs as policies or world models.
- **Neuro-Symbolic MARL**: Combining deep learning with symbolic reasoning.
- **Human-Agent Collaboration**: Agents that learn to assist and communicate with humans.
- **Real-World Deployment**: Closing the sim-to-real gap for physical multi-agent systems.


In [ ]:
# Limitations vs Future Directions comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))

# Limitations
limitations = ['Sample\nEfficiency', 'Scalability', 'Generalization', 'Safety', 'Evaluation\nStandards']
severity = [95, 90, 85, 80, 75]
colors_lim = ['#e74c3c', '#e67e22', '#f39c12', '#f1c40f', '#f39c12']

bars1 = ax1.bar(limitations, severity, color=colors_lim, alpha=0.8, edgecolor='black', linewidth=1.5)
ax1.set_ylim(0, 110)
ax1.set_ylabel('Challenge Severity', fontsize=12, fontweight='bold')
ax1.set_title('⚠️ Current Limitations in MARL', fontsize=14, fontweight='bold', color='#c0392b')
ax1.grid(True, axis='y', alpha=0.3)
for bar, val in zip(bars1, severity):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, f'{val}%', 
             ha='center', fontsize=10, fontweight='bold')

# Future directions
directions = ['Foundation\nModels', 'Neuro-\nSymbolic', 'Human-Agent\nCollab', 'Sim-to-\nReal', 'Multi-Modal\nAgents']
potential = [95, 85, 90, 80, 88]
colors_fut = ['#2ecc71', '#27ae60', '#1abc9c', '#16a085', '#2980b9']

bars2 = ax2.bar(directions, potential, color=colors_fut, alpha=0.8, edgecolor='black', linewidth=1.5)
ax2.set_ylim(0, 110)
ax2.set_ylabel('Research Potential / Impact', fontsize=12, fontweight='bold')
ax2.set_title('🔮 Future Research Directions', fontsize=14, fontweight='bold', color='#27ae60')
ax2.grid(True, axis='y', alpha=0.3)
for bar, val in zip(bars2, potential):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, f'{val}%', 
             ha='center', fontsize=10, fontweight='bold')

plt.suptitle('MARL: Challenges and Opportunities', fontsize=18, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


## 🛠️ Hands-On Exercises

Complete these exercises to solidify your understanding of Multi-Agent RL concepts.


### Exercise 1: Implement a Simple Two-Agent Cooperative Environment
Create a new grid world where two agents must push a box to a target location. Both agents must be adjacent to the box to move it.


### Exercise 2: Train Independent Q-Learning Agents
Use the IQL implementation from this notebook to train agents in your new environment. Plot the learning curves.


### Exercise 3: Observe Non-Stationarity Issues
Train one agent while keeping another's policy fixed, then switch. Document how performance drops when the stationary agent's behavior changes.


### Exercise 4: Experiment with Shared vs Individual Rewards
Modify the reward structure to give individual rewards based on distance to goal vs shared reward only when both succeed. Compare convergence.


### Exercise 5: Implement a Competitive Scenario
Create a zero-sum game where one agent tries to catch another (pursuit-evasion). Train both agents using IQL.


### Exercise 6: Visualize Joint Action Q-Values
For a small state in your environment, compute and visualize the Q-values for all joint action combinations as a heatmap.


### Exercise 7: Discuss How to Scale to More Agents
Write a brief analysis (in markdown or comments) on what changes would be needed to scale your environment to 5+ agents. Consider communication and CTDE.


### Exercise 8: Explore Hierarchical RL Ideas
Design a two-level hierarchy for your multi-agent task: a high-level manager that selects sub-goals and low-level workers that execute primitive actions.


### Exercise 9: Suggest Real-World Applications for MARL
Identify a real-world problem in your domain of interest that could benefit from MARL. Describe the agents, environment, reward structure, and challenges.


### Exercise 10: Research One Modern MARL Algorithm
Choose QMIX, MADDPG, or MAPPO. Write a summary of how it works and why it addresses specific MARL challenges. Include pseudocode if helpful.


## Solutions & Discussion (Review After Attempting)


### Solution 1: Two-Agent Cooperative Environment
**Key Design Principles:**
- State space should include positions of both agents AND the box.
- Transition function must check if both agents are adjacent to the box before allowing movement.
- Reward should be sparse (only when box reaches target) to encourage true cooperation.

**Implementation Hint:**
```python
def step(self, action1, action2):
    # Move agents independently
    # Check if box can be moved (both agents adjacent and pushing in same direction)
    # Update positions and return state, reward, done
```


### Solution 2: Training IQL Agents
**Expected Behavior:**
- Early episodes: Agents wander randomly, rarely achieve joint success.
- Mid training: Agents learn to approach the box but may not coordinate pushes.
- Late training: If successful, agents learn to position themselves and push simultaneously.

**Common Pitfalls:**
- Too high learning rate causes oscillation.
- Too much exploration decay prevents finding cooperative strategy.
- Sparse rewards make credit assignment difficult.


### Solution 3: Non-Stationarity
**Observation:** When Agent 1's policy changes, Agent 2's Q-values become outdated because they were learned under Agent 1's old behavior. This manifests as:
- Sudden drops in reward when policies change.
- Need for continual adaptation (unlike stationary single-agent environments).
- Justification for CTDE: centralized training stabilizes learning by providing global information.


### Solution 4: Shared vs Individual Rewards
**Analysis:**
| Reward Type | Pros | Cons |
|-------------|------|------|
| **Shared** | Encourages cooperation, simple | Credit assignment problem, lazy agent problem |
| **Individual (distance)** | Clear credit, faster initial learning | May lead to competition, suboptimal global behavior |
| **Mixed (shaped)** | Best of both worlds | Requires domain knowledge to design |


### Solution 5: Competitive Pursuit-Evasion
**Environment Design:**
- Evader gets +1 for each step not caught, -10 if caught.
- Pursuer gets +10 for catching, -0.1 per step.
- This is zero-sum if pursuer reward = -evader reward.

**Training Challenge:**
- Self-play is common: train pursuer against current best evader and vice versa.
- Leads to iterative policy improvement similar to GAN training.


### Solution 6: Joint Action Q-Values
**Visualization Approach:**
For a state where both agents have 4 actions each, create a 4x4 heatmap:
```python
joint_q = np.zeros((4, 4))
for a1 in range(4):
    for a2 in range(4):
        # This requires a joint Q-table: Q[state][a1][a2]
        joint_q[a1, a2] = joint_q_table[state][a1][a2]
sns.heatmap(joint_q, annot=True, fmt='.2f')
```
The maximum value indicates the optimal joint action.


### Solution 7: Scaling to More Agents
**Key Considerations:**
1. **Communication**: Implement a communication channel where agents broadcast vectors (CommNet, TarMAC).
2. **Attention**: Use attention mechanisms to focus on relevant agents (MAT, MAAC).
3. **Mean Field**: Approximate interactions with average neighborhood effect (MADDPG → Mean Field AC).
4. **Graph Neural Networks**: Model agents as nodes in a graph for scalable message passing.


### Solution 8: Hierarchical RL Design
**Two-Level Architecture:**
- **Manager**: Operates at low frequency, selects sub-goal states (e.g., "go to box", "push box north").
- **Worker**: Operates at high frequency, executes primitive actions to achieve sub-goal.
- **Intrinsic Reward**: Worker receives reward for achieving manager's sub-goal.

This decomposes the credit assignment problem and enables transfer learning across tasks.


### Solution 9: Real-World Application Example
**Smart Traffic Management:**
- **Agents**: Traffic lights at intersections.
- **Environment**: Road network with vehicle flows.
- **Reward**: Negative of average wait time / throughput.
- **Challenges**: Non-stationary traffic patterns, partial observability, safety constraints, real-time requirements.


### Solution 10: QMIX Deep Dive
**Core Idea:** Factorize joint action-value function $Q_{tot}$ using individual $Q_i$ values and a mixing network.

**Key Properties:**
- **Monotonicity**: $\frac{\partial Q_{tot}}{\partial Q_i} > 0$ ensures decentralized policies are consistent with centralized optimum.
- **State-Dependent Mixing**: Mixing weights are generated by hypernetworks conditioned on global state.
- **CTDE**: Train with global state, execute with local observations only.

**Why it works:**
Addresses non-stationarity by providing stable centralized training while avoiding exponential joint action spaces during execution.
